# Model Comparison: Exact versus approximate solutions

This notebook gives an overview of the differences between the `Mibitrans`, the `Anatrans` and the `Bioscreen` models, which reflect the use of a fully exact and fully analytical, but approximate solutions to the ADE for modelling 
contaminant transport. Guyonnet and Neville (2004)$^1$ studies the differences (originating from an integral approximation) and its causes. This notebook visualizse the amount and effects of the error. 

**Authors:** Alraune Zech, Jorrit Bakker

## Background

The `Mibitrans` model classes uses the exact solution of the 3D transport ADE, as described in Wexler et al. (1992) which requires numerical integration. The `Anatrans model` uses an approximate but fully analytical expression. Both models include the option for multiple source zones and source depletion.  

### Mibitrans model
The equation implemented in the `Mibitrans` model reads:
$$
\begin{equation}\tag{1}
\begin{aligned}
    C(x,y,t) &= \sum_{i=1}^{n}\left\{C^*_{0,i}\frac{x}{8\sqrt{\pi \alpha_{x}v_R}}\exp(-\gamma_s t) \cdot  \right. \\
    &\quad\int_{0}^{t}\frac{1}{\tau^{\frac{3}{2}}} \exp\left((\gamma_s - \mu)\tau - \frac{(x-v_R\tau)^2}{4\alpha_{x}v_R\tau} \right) \cdot \\
    &\quad \left[\operatorname{erfc}\left(\frac{y-Y_i}{2 \sqrt{\alpha_{y} v_R\tau}}\right)-\operatorname{erfc}\left(\frac{y+Y_i}{2 \sqrt{\alpha_{y}v_R\tau}}\right) \right]\cdot  \\
    &\quad  \left. \left[\operatorname{erfc}\left(\frac{-Z}{2 \sqrt{\alpha_{z}v_R\tau}}\right)-\operatorname{erfc}\left(\frac{Z}{2 \sqrt{\alpha_{z}v_R\tau}}\right) \right] d\tau \right\}
\end{aligned}
\end{equation}
$$
Here: $v_R = \frac{v}{R}$ is the retarded velocity of contaminant transport. Other parameters, see Parameter overview table in Documentation.

### Anatrans model

The equation implemented in the `Anatrans` model reads:

$$
\begin{equation}\tag{2}
\begin{aligned}
    C(x,y,t) &= \sum_{i=1}^{n}\left\{ \frac{C^*_{0,i}}{8} \exp \left(-\gamma_s t\right)\cdot \right.  \\ 
    &\quad \left[\exp \left( \frac{x\left(1-\tilde P\right)}{2\alpha_x}\right) \right. \cdot \operatorname{erfc} \left( \frac{x - v_Rt\tilde P}{2\sqrt{\alpha_x v_Rt }} \right)\\
    &\quad \ \, +  \left.\exp \left( \frac{x\left(1+\tilde P\right)}{2\alpha_x}\right) \cdot \operatorname{erfc} \left( \frac{x + v_Rt\tilde P}{2\sqrt{\alpha_x v_Rt }} \right) \right]\cdot \\
    &\quad \left[ \operatorname{erf} \left( \frac{y + Y_i}{2\sqrt{\alpha_y x}} \right) - \operatorname{erf} \left( \frac{y - Y_i}{2\sqrt{\alpha_y x)}} \right) \right]\cdot \\
    &\quad  \left. \left[ \operatorname{erf} \left( \frac{Z}{2\sqrt{\alpha_z x)}} \right) - \operatorname{erf} \left( \frac{-Z}{2\sqrt{\alpha_z x}} \right) \right] \right\} 
\end{aligned}
\end{equation}
$$

with $\tilde P = \sqrt{1+4 (\mu-\gamma_s) \alpha_x/v_R}$

The `Anatrans` model equation can be derived from the exact `Mibitrans` model solution by splitting the effect of time on the different directions following the principle $C(x,y,z,t)/C_0 = C(x,t)\cdot C(y,x)\cdot C(z,x)/C_0$. Specifically, by applying the substitution $\tau = x/v_R$ (time traveled of advection front) in the error-function terms for the $y$ and $z$ directions. The remaining integral in $\tau$ in  the `Mibitrans` model for the transport in $x$-direction can be solved analytically resulting in the closed form expression in the `Anatrans` model.

### Bioscreen model

The equation implemented in the `Bioscreen` model reads:
$$
\begin{equation}\tag{3}
\begin{aligned}
    C(x, y, t) & = \sum_{i=1}^{n} \left\{\frac{C^*_{0,i}}{8} \exp{\left( -\gamma_s \left( t-\frac{x}{v_R}\right)\right)} \cdot \right. \\ 
    &\quad \exp \left( \frac{x\left(1-P\right)}{2\alpha_x}\right)
    \operatorname{erfc} \left(\frac{x - Pv_Rt}{2\sqrt{\alpha_x v_Rt }} \right) \cdot \\
    &\quad  \left[ \operatorname{erf} \left( \frac{y + Y_i}{2\sqrt{\alpha_y x}} \right) - \operatorname{erf} \left( \frac{y - Y_i}{2\sqrt{\alpha_y x)}} \right) \right] \cdot \\
    & \quad \left.\left[ \operatorname{erf} \left(\frac{Z}{2\sqrt{\alpha_z x)}} \right) - \operatorname{erf} \left( \frac{-Z}{2\sqrt{\alpha_z x}} \right) \right] \right\} 
\end{aligned}
\end{equation}
$$
with $P = \sqrt{1+4 \mu \alpha_x/v_R}$.


### Approximation
The approximation for transverse dispersivity terms in the `Anatrans` model equation introduces an error. The size of the error depends on parameter choices. Guyonnet and Neville (2004)$^1$ studies the differences and its causes in more detail, using a dimensionless analysis. This notebook allows to visualize the amount and effects of the error introduced by the integral approximation. 

### Model Comparison Parameters

| **Quantity (Unit)** | **Values** |
|---|---|
| α_L (m) | 1; 3; **10** |
| α_T (m) | 0.002; **0.02**; 0.2 |
| α_V (m) | 0; 0.0005; **0.005**; 0.05 |
| Source zone boundaries, Y_i (m) | 1; **5**; 20 |


### References
$^1$ [Guyonnet, D., & Neville, C. (2004). *Dimensionless analysis of two analytical solutions for 3-D solute transport in groundwater*. Journal of Contaminant Hydrology, 75(1), 141–153.](https://doi.org/10.1016/j.jconhyd.2004.06.004)


## Qualitative Model comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import mibitrans as mbt

In [ ]:
###  Functions for quantitative comparison
import differences as df

In [ ]:
### Define options for plot modification
cmap = plt.get_cmap('tab20b')
colors_1 = cmap.colors   # list of RGB tuples
cmap = plt.get_cmap('Set1')
colors_2 = cmap.colors   # list of RGB tuples

### Model Setup and Parameter Definition

In [ ]:
x_disp = np.array([10,3,1])                   #[m]
y_disp = np.array([0.2,0.02,0.002])           #[m]
z_disp = np.array([0.05, 0.005, 0.0005, 0])   #[m]

**Define Imput parameters for standard values of dispersivity**

Other standard settings:
* no decay
* standard example values

In [ ]:
hydro = mbt.HydrologicalParameters(
    velocity=0.1,      # Flow velocity [m/d]
    porosity=0.3,      # Effective soil porosity [-]
    alpha_x=10,        # Longitudinal dispersivity, in [m]
    alpha_y=0.02,      # Transverse horizontal dispersivity, in [m]
    alpha_z=0.005,     # Transverse vertical dispersivity, in [m]
    diffusion=0,       # Molecular diffusion, in [m2/day]
)

att = mbt.AttenuationParameters(
    retardation=1,
    half_life=0,
)

source = mbt.SourceParameters(
    source_zone_boundary=5,        #[m]
    source_zone_concentration=10,  #[g/m3]
    depth=2,                       #[m]
    total_mass=np.inf              #[g]
)

model = mbt.ModelParameters(
    model_length = 600,
    model_width = 40,
    model_time = 10*365,
    dx = 5,
    dy = 0.25,
    dt = 365//5,
)

In [ ]:
mbt_object = mbt.Mibitrans(hydro, att, source, model)
mbt_results = mbt_object.run()

ana_object = mbt.Anatrans(hydro, att, source, model)
ana_results = ana_object.run()

bio_model = mbt.Bioscreen(hydro, att, source, model)
bio_results = bio_model.run()

In [ ]:
# Plot the x and y concentration distribution for no degradation decay model, uses plot_surface
mbt_results.plume_3d()#time=5*365)
plt.show()

### Visual comparison for Standard Parameters

In [ ]:
t1 = 365 #1y
t2 = 3*365 #3y
t3 = 9*365 # 9y

mbt_results.centerline(time=t1, color=colors_1[0],ls = '-.',lw=3, label="Mibitrans: $t_1$")
ana_results.centerline(time=t1, color=colors_1[0+8],ls = '-.',lw=2.5, label="Anatrans: $t_1$")
bio_results.centerline(time=t1, color=colors_1[0+12],ls = '-.',lw=2.5, label="Bioscreen: $t_1$")

mbt_results.centerline(time=t2, color=colors_1[1],ls = '--',lw=2.5, label="$t_2$")
ana_results.centerline(time=t2, color=colors_1[1+8],ls = '--',lw=2.5, label="$t_2$")
bio_results.centerline(time=t2, color=colors_1[1+12],ls = '--',lw=2.5, label="$t_2$")

mbt_results.centerline(time=t3, color=colors_1[2],ls = '-',lw=2.5, label="$t_3$")
ana_results.centerline(time=t3, color=colors_1[2+8],ls = '-',lw=2.5, label="$t_3$")
bio_results.centerline(time=t3, color=colors_1[2+12],ls = '-',lw=2.5, label="$t_3$")

plt.legend(ncols = 3)
plt.show()

In [ ]:
# plt.figure(figsize = [3.75,2.75])
xx = [20,110,350]

mbt_results.transverse(x_position = xx[0], time=t1, color=colors_1[0],ls = '-.',lw=2, label="Mibitrans, $t_1$")
ana_results.transverse(x_position = xx[0], time=t1, color=colors_1[0+8],ls = '-.',lw=2, label="Anatrans, $t_1$")
bio_results.transverse(x_position = xx[0], time=t1, color=colors_1[0+12],ls = '-.',lw=2, label="Bioscreen, $t_1$")

mbt_results.transverse(x_position = xx[1], time=t2, color=colors_1[1],ls = '--',lw=2, label="Mibitrans, $t_2$")
ana_results.transverse(x_position = xx[1], time=t2, color=colors_1[1+8],ls = '--',lw=2, label="Anatrans, $t_2$")
bio_results.transverse(x_position = xx[1], time=t2, color=colors_1[1+12],ls = '--',lw=2, label="Bioscreen, $t_2$")

mbt_results.transverse(x_position = xx[2], time=t3, color=colors_1[2],ls = '-',lw=2, label="Mibitrans, $t_3$")
ana_results.transverse(x_position = xx[2], time=t3, color=colors_1[2+8],ls = '-',lw=2, label="Anatrans, $t_3$")
bio_results.transverse(x_position = xx[2], time=t3, color=colors_1[2+12],ls = '-',lw=2, label="Bioscreen, $t_3$")
plt.show()

### Quantitative Comparison for Standard Parameters

In [ ]:
diff_mbt_bio = df.mean_absolute_difference(bio_results.relative_cxyt[:,:,:], mbt_results.relative_cxyt[:,:,:])
diff_mbt_ana = df.mean_absolute_difference(ana_results.relative_cxyt[:,:,:], mbt_results.relative_cxyt[:,:,:])
diff_ana_bio = df.mean_absolute_difference(bio_results.relative_cxyt[:,:,:], ana_results.relative_cxyt[:,:,:])

print("Mean absolute difference in relative concentration with standard example parameters (no decay):")
print("Between Mibitrans and Bioscreen model class",  diff_mbt_bio)
print("Between Mibitrans and Anatrans model class",  diff_mbt_ana)
print("Between Anatrans and Bioscreen model class",  diff_ana_bio)

In [ ]:
diff_mbt_bio_t = df.mean_absolute_difference(bio_results.relative_cxyt[:,:,:], mbt_results.relative_cxyt[:,:,:],
                                             mean_axis = (1,2))
diff_mbt_ana_t = df.mean_absolute_difference(ana_results.relative_cxyt[:,:,:], mbt_results.relative_cxyt[:,:,:],
                                             mean_axis = (1,2))
diff_ana_bio_t = df.mean_absolute_difference(bio_results.relative_cxyt[:,:,:], ana_results.relative_cxyt[:,:,:],
                                             mean_axis = (1,2))
#print(len(diff_mbt_bio_t), len(mbt_results.t))

In [ ]:
print("Maximum mean relative difference between Mibitrans and Bioscreen model:", np.max(diff_mbt_bio_t))
print("Maximum mean relative difference between Mibitrans and Anatrans model:", np.max(diff_mbt_ana_t))
print("Maximum mean relative difference between Anatrans and Bioscreen model:", np.max(diff_ana_bio_t))

In [ ]:
plt.plot(mbt_results.t/365, diff_mbt_bio_t*100,
         marker = '.', color = colors_2[4], ls = '-', label = 'Mibitrans vs Bioscreen')
plt.plot(mbt_results.t/365, diff_ana_bio_t*100,
         marker = '.', color = colors_2[3], ls = '-', label = 'Anatrans vs Bioscreen')
plt.plot(mbt_results.t/365, diff_mbt_ana_t*100,
         marker = '.', color = colors_2[2], ls = '-', label = 'Mibitrans vs Anatrans')
plt.legend()
plt.show()

In [ ]:
time_step = 14 #len(mbt_object.t)//2 - 1
print(time_step,mbt_object.t[time_step]/365)
print(mbt_object.disp_x,mbt_object.disp_y,mbt_object.disp_z)

diff_mbt_bio = bio_results.relative_cxyt[time_step,:,:] - mbt_results.relative_cxyt[time_step,:,:]
diff_mbt_ana = ana_results.relative_cxyt[time_step,:,:] - mbt_results.relative_cxyt[time_step,:,:]

In [ ]:
plt.pcolormesh(
    mbt_object.x[:len(mbt_object.x)//4],
    mbt_object.y,
    diff_mbt_bio[:,:len(mbt_object.x)//4],
    vmin=-0.2,
    vmax=0.2,
    cmap="seismic",
    shading="gouraud"
)
cbar = plt.colorbar(orientation="vertical")
cbar.set_label("Absolute concentration difference")
plt.title("Concentration difference in space: Mibitrans vs Bioscreen")
plt.xlabel("Plume travel distance $x$ [m]")
plt.ylabel('Plume cross section $y$ [m]')
plt.ylim((-10,10))

In [ ]:
plt.pcolormesh(
    mbt_object.x[:len(mbt_object.x)//4],
    mbt_object.y,
    diff_mbt_ana[:,:len(mbt_object.x)//4],
    vmin=-0.2,
    vmax=0.2,
    cmap="seismic",
    shading="gouraud"
)

cbar = plt.colorbar(orientation="vertical")
cbar.set_label("Absolute concentration difference")
plt.title("Concentration difference in space: Mibitrans vs Anatrans")
plt.xlabel("Plume travel distance $x$ [m]")
plt.ylabel('Plume cross section $y$ [m]')
plt.ylim((-10,10))

## Quantitative Model comparison & Parameter Sensitivity

In [ ]:
att = mbt.AttenuationParameters(
    retardation=1.3,      #[m]
    half_life=5*365     #[days]
)

model = mbt.ModelParameters(
    model_length = 400,    #[m]
    model_width = 60,      #[m]
    model_time = 10*365,    #[m]
    dx = 2,                 #[m]
    dy = 0.5,                 #[m]
    dt = 365                #[days]
)

Run the models, output is a nested list, indexed as [z][y][x], in the order of the dispersivity arrays defined above. Thus, in this example, `list[2][0][1]` would give the model results with $\alpha_x = 3m$, $\alpha_y = 0.2m$, $\alpha_z = 0.0005m$.

In [ ]:
list_mbt, list_ana, list_bio = df.sensitivity_models(hydro, att, source, model, x_disp, y_disp, z_disp)


Then compare the two models using the absolute difference between relative concentrations.

In [ ]:
df.comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 0, # Concentration value below which a value will not be included in the average
    relative_concentration = True, # If True, differences are calculated from relative concentrations C/C0.
    difference_method = "absolute", # 'relative', 'absolute' or 'rmse'
    relative_diff_diff = False, # If True, largest value will be set to 1, other values are scaled proportionally
    as_percentage = True, # If True, multiply all values by 100, prevents too much 0's behind the decimal.
    colorbar_label = "mean absolute difference in relative concentration [%]"
)

These results fall within what is expected based on theory, longitudinal dispersivity has a large influence on the deviation of the Anatrans solution from the Mibitrans solution. If transverse horizontal dispersivity is high, transverse vertical dispersivity has a relatively small influence, although its influence increases for lower transverse horizontal dispersivities. 

Taking the average across the entire model domain is not a valid comparison method. This way includes a large amount of 0 values, either because the advective front has not progressed enough, or due to the plume not dispersing as far as the model boundary. An alternative is to set a cutoff value, below which cells are not included in calculation of the mean difference.

In [ ]:
df.comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 1e-3,
    relative_concentration = True,
    difference_method = "absolute",
    relative_diff_diff = False,
    as_percentage = True,
    colorbar_label = "mean absolute difference in relative concentration [%]"
)

Although the pattern in effect of longitudinal dispersivity is preserved, the effect of transverse horizontal dispersivity is contrary to the previous plot, and contrary to what is expected from the theory. If the Anatrans model is equal to the Mibitrans model if $\alpha_y = \alpha_z = 0$, one would expect that with $\lim_{\alpha_y, \alpha_z \to 0}$ , differences between the models should approach 0 as well. However, if only $\alpha_y \to 0$, the error introduced by the transverse vertical term still remains. And since the plume spreads little in the transverse horizontal direction, all difference is contained in a small area, leading to a high average. On the contrary, For higher values of transverse horizontal dispersivity, differences might 'add' up to more in total, but since they are spread out over a larger area, they average out to less. Therefore, the values from this method are more reflective of the general behaviour of the 3D-ADE, than of the sensitivity of the approximate solution to dispersivity.

This is visualized in the graphs below.

In [ ]:
time = 5*365
models=[list_mbt[0][-1][0], list_mbt[0][0][0],
        list_ana[0][-1][0], list_ana[0][0][0],
        list_bio[0][-1][0], list_bio[0][0][0]]
colors=["cornflowerblue", "blue",
        "gold", "goldenrod",
        "coral", "red"]
linewidth=2
linestyles=["-", "-", "--", "--", "-.", "-."]
labels=[r"mbt, $\alpha_y=0.002m$", r"mbt, $\alpha_y=0.2m$",
        r"ana, $\alpha_y=0.002m$", r"ana, $\alpha_y=0.2m$",
        r"bio, $\alpha_y=0.002m$", r"bio, $\alpha_y=0.2m$"]

for i, mod in enumerate(models):
    mod.centerline(time=time, color=colors[i], lw=linewidth, linestyle=linestyles[i], label=labels[i])
plt.legend(ncols=3)
plt.xlim(-10, 275)
plt.show()

xpos = 100
for i, mod in enumerate(models):
    mod.transverse(x_position=xpos, time=time, color=colors[i], lw=linewidth, linestyle=linestyles[i], label=labels[i])
plt.legend()
plt.xlim(-20,20)
plt.show()

In [ ]:
### Specification for Plot
models=[list_mbt[0][-1][0],list_mbt[0][0][0], list_mbt[-1][0][0],
        list_ana[0][-1][0],list_ana[0][0][0], list_ana[-1][0][0], 
        list_bio[0][-1][0],list_bio[0][0][0], list_bio[-1][0][0]]
#az = 0.05, ay = 0.002, ax = 10 : [0][-1][0]
#az = 0.05, ay = 0.2, ax = 10: [0][0][0]
#az = 0, az = 0.2, ax = 10: [-1][0][0]
labels=[r"mbt, $\alpha_y=0.002m$", r"mbt, $\alpha_y=0.2m$", r"mbt, $\alpha_z=0m$",
        r"ana", r"ana", r"ana", r"bio", r"bio", r"bio"]
linestyles=["-", "--",':', "-", "--",':', "-", "--",':']
colors=[colors_1[0],colors_1[1],colors_1[2], 
        colors_1[0+8],colors_1[1+8],colors_1[2+8],
        colors_1[0+12],colors_1[1+12],colors_1[2+12]]

In [ ]:
for i, mod in enumerate(models):
    mod.transverse(x_position=100, time=5*365, color=colors[i], lw=2, linestyle=linestyles[i], label=labels[i])

plt.xlim(-20,20)
plt.ylim([0,5])
plt.legend(ncols = 3)#, loc= 'lower left')#bbox_to_anchor=(0.3,0.55))
# plt.xlabel('Plume cross section position $y$ [m]',fontsize = textsize)
# plt.ylabel('Concentration $C(x_i,y,t_i)$',fontsize = textsize)
# plt.title("")#Plume mass over time")
# #plt.xlim([0,500])
# plt.grid(True)

Another possibility is to use the relative differences between models. However, the smaller values at the plume fringes tend to have larger relative errors, even though the absolute difference at these positions would be too small to be significant for most practical purposes.

In [ ]:
df.comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 1e-5,
    relative_concentration = True,
    difference_method = "relative",
    relative_diff_diff = False,
    as_percentage = False,
    colorbar_label = "mean relative difference in concentration"
)

Furthermore, differences now seem highest for the lowest values of $\alpha_z$, even though absolute differences are higher for the higher values of $\alpha_z$, as shown below.

In [ ]:
models=[list_mbt[-1][0][0], list_mbt[0][0][0],
        list_ana[-1][0][0], list_ana[0][0][0],
        list_bio[-1][0][0], list_bio[0][0][0]]
colors=["cornflowerblue", "blue",
        "gold", "goldenrod",
        "coral", "red"]
linestyles=["-", "-", "--", "--", "-.", "-."]
labels=[r"mbt, $\alpha_z=0m$", r"mbt, $\alpha_z=0.05m$",
        r"ana, $\alpha_z=0m$", r"ana, $\alpha_z=0.05m$",
        r"bio, $\alpha_z=0m$", r"bio, $\alpha_z=0.05m$"]

for i, mod in enumerate(models):
    mod.centerline(time=5*365, color=colors[i], lw=2,
                   linestyle=linestyles[i], label=labels[i])
plt.legend()
plt.xlim(-10, 275)
plt.show()

xpos = 100
for i, mod in enumerate(models):
    mod.transverse(x_position=xpos, time=5*365, color=colors[i],
                   lw=2, linestyle=linestyles[i], label=labels[i])
plt.legend()
plt.xlim(-20,20)
plt.show()

This tenancy of the relative error can be somewhat circumvented by choosing a higher cutoff value, preventing low concentrations from dominating the average. Below, 2% of the source concentration is taken as the cutoff value.

In [ ]:
df.comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 2e-2,
    relative_concentration = True,
    difference_method = "relative",
    relative_diff_diff = False,
    as_percentage = False,
    colorbar_label = "mean relative difference in concentration"
)

In [ ]:
df.comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 2e-2,
    relative_concentration = True,
    difference_method = "relative",
    relative_diff_diff = False,
    as_percentage = False,
    colorbar_label = "mean relative difference in concentration"
)

However, this makes the choice of the cutoff value rather arbitrary, where how well the resulting values reflect your expectations depend on which value is chosen. Furthermore, for high source concentrations, or higihly toxic contaminants, concentrations below the increased cutoff value might still be significant.

The issues with these methods can be prevented by applying the concentration cutoff in the longitudinal direction only (which eliminates places which the advective front has not yet reached). Then taking the sum of the differences in the transverse horizontal direction, preventing averaging over different domains for different dispersivities. Then taking the mean in the x and t direction in respective order. This last step prevents errors from the earlier time steps, where the advective front has not progressed as far and thus would contribute less to the overall mean, from being underrepresented. This represents differences between the models quite well, however, the values calculated lose their physical meaning. Therefore, it might be better to the differences relative to each other, with 1 being the highest difference among the dispersivities and the rest scaled proportionally (accomplished by setting relative_diff_diff to True). Though this last step is optional. Note how this method is not as sensitive to the choice of cutoff value as the other methods.

In [ ]:
df.comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 1e-4,
    relative_concentration = True,
    difference_method = "sum_mean",
    relative_diff_diff = True,
    as_percentage = False,
    colorbar_label = "normalized difference between models"
)

In [ ]:
df.comparison_plot(
    list_mbt, list_bio, x_disp, y_disp, z_disp,
    cutoff = 1e-4,
    relative_concentration = True,
    difference_method = "sum_mean",
    relative_diff_diff = True,
    as_percentage = False,
    colorbar_label = "normalized difference between models"
)

Note: The latter method seems to somewhat overestimate the size in difference for smaller longitudinal dispersivities

In [ ]:
time = 5*365
models=[list_mbt[0][0][0], list_mbt[0][0][-1],
        list_ana[0][0][0], list_ana[0][0][-1],
        list_bio[0][0][0], list_bio[0][0][-1]]
colors=["blue","cornflowerblue",
        "goldenrod", "gold",
        "red", "coral"]
linewidth=2
linestyles=["-", "-", "--", "--", "-.", "-."]
labels=[r"mbt, $\alpha_z=0m$", r"mbt, $\alpha_z=0.05m$",
        r"ana, $\alpha_z=0m$", r"ana, $\alpha_z=0.05m$",
        r"bio, $\alpha_z=0m$", r"bio, $\alpha_z=0.05m$"]

for i, mod in enumerate(models):
    mod.centerline(time=time, color=colors[i], lw=linewidth, linestyle=linestyles[i], label=labels[i])
plt.legend()
plt.xlim(-10, 275)
plt.show()

xpos = 100
for i, mod in enumerate(models):
    mod.transverse(x_position=xpos, time=time, color=colors[i], lw=linewidth, linestyle=linestyles[i], label=labels[i])
plt.legend()
plt.xlim(-20,20)
plt.show()

Regardless, it is difficult and slightly misleading to represent the differences between models as a single value, since differences are...

## Paper

In [ ]:
textsize = 8

In [ ]:
times = mbt_results.t[4::20]
#print(times)
t1 = 365 #1y
t2 = 3*365 #3y
t3 = 9*365 # 9y
lw1,lw2,lw3 = 2.5,2.5,2.5

plt.figure(figsize = [3.75,2.75])
mbt_results.centerline(time=t1, color=colors_1[0],ls = '-.',lw=lw1+0.5, label="Mibitrans: $t_1$")
ana_results.centerline(time=t1, color=colors_1[0+8],ls = '-.',lw=lw2, label="Anatrans: $t_1$")
bio_results.centerline(time=t1, color=colors_1[0+12],ls = '-.',lw=lw3, label="Bioscreen: $t_1$")

mbt_results.centerline(time=t2, color=colors_1[1],ls = '--',lw=lw1, label="$t_2$")
ana_results.centerline(time=t2, color=colors_1[1+8],ls = '--',lw=lw2, label="$t_2$")
bio_results.centerline(time=t2, color=colors_1[1+12],ls = '--',lw=lw3, label="$t_2$")

mbt_results.centerline(time=t3, color=colors_1[2],ls = '-',lw=lw1, label="$t_3$")
ana_results.centerline(time=t3, color=colors_1[2+8],ls = '-',lw=lw2, label="$t_3$")
bio_results.centerline(time=t3, color=colors_1[2+12],ls = '-',lw=lw3, label="$t_3$")

plt.legend(ncols = 3,fontsize = textsize,bbox_to_anchor=(0.2,0.85))
#plt.legend(ncols = 3,loc = 'upper right',fontsize = textsize)
plt.xlim((-1,420)) # Reduce the x-axis limit to better observe differences
plt.ylim((0,11))
plt.xlabel('Plume travel distance $x$',fontsize = textsize)
plt.ylabel('Concentration $C(x,y=0,t_i)$',fontsize = textsize)
plt.grid(True)
plt.title("")#Plume mass over time")
plt.tick_params(axis='both', labelsize=textsize)
plt.tight_layout()
#plt.savefig('model_comparison_centerline.pdf')

In [ ]:
xx = [20,110,350]
lw = 2.5

plt.figure(figsize = [3.75,2.75])
mbt_results.transverse(x_position = xx[0], time=t1, color=colors_1[0],ls = '-.',lw=lw, label="Mibitrans, $t_1$")
ana_results.transverse(x_position = xx[0], time=t1, color=colors_1[0+8],ls = '-.',lw=lw, label="Anatrans, $t_1$")
bio_results.transverse(x_position = xx[0], time=t1, color=colors_1[0+12],ls = '-.',lw=lw, label="Bioscreen, $t_1$")

mbt_results.transverse(x_position = xx[1], time=t2, color=colors_1[1],ls = '--',lw=lw, label="Mibitrans, $t_2$")
ana_results.transverse(x_position = xx[1], time=t2, color=colors_1[1+8],ls = '--',lw=lw, label="Anatrans, $t_2$")
bio_results.transverse(x_position = xx[1], time=t2, color=colors_1[1+12],ls = '--',lw=lw, label="Bioscreen, $t_2$")

mbt_results.transverse(x_position = xx[2], time=t3, color=colors_1[2],ls = '-',lw=lw, label="Mibitrans, $t_3$")
ana_results.transverse(x_position = xx[2], time=t3, color=colors_1[2+8],ls = '-',lw=lw, label="Anatrans, $t_3$")
bio_results.transverse(x_position = xx[2], time=t3, color=colors_1[2+12],ls = '-',lw=lw, label="Bioscreen, $t_3$")

#plt.legend(bbox_to_anchor=(1,1),ncols = 3)
plt.ylim((0, 9))
plt.xlim((-10,10))
plt.grid(True)
plt.title("")#Plume mass over time")

plt.xlabel('Plume cross section $y$',fontsize = textsize)
plt.ylabel('Concentration $C(x_i,y,t_i)$',fontsize = textsize)

plt.tick_params(axis='both', labelsize=textsize)
plt.tight_layout()
#plt.savefig('model_comparison_trans.pdf')

In [ ]:
plt.figure(figsize = [3.75,2.5])
plt.plot(mbt_results.t/365, diff_mbt_bio_t*100,
         marker = '.', color = colors_2[4], ls = '-', label = 'Mibitrans vs Bioscreen')
plt.plot(mbt_results.t/365, diff_ana_bio_t*100,
         marker = '.', color = colors_2[3], ls = '-', label = 'Anatrans vs Bioscreen')
plt.plot(mbt_results.t/365, diff_mbt_ana_t*100,
         marker = '.', color = colors_2[2], ls = '-', label = 'Mibitrans vs Anatrans')

plt.legend(fontsize = textsize)
plt.xlabel('Travel time of Plume $t$ (years)',fontsize = textsize)
plt.ylabel('Mean relative difference in %',fontsize = textsize)
plt.xlim([0,10])
plt.grid(True)
plt.tick_params(axis='both', labelsize=textsize)
plt.tight_layout()
#plt.savefig('model_comparison_diff_t.pdf')

In [ ]:
plt.figure(figsize = [3.75,2.5])

plt.pcolormesh(
    # mbt_object.x,
    # mbt_object.y,
    # diff_mbt_bio,
    mbt_object.x[:len(mbt_object.x)//4],
    mbt_object.y,
    diff_mbt_bio[:,:len(mbt_object.x)//4],
    # mbt_object.x[:len(mbt_object.x)//4],
    # mbt_object.y[len(mbt_object.y)//5:-len(mbt_object.y)//5],
    # diff_mbt_bio[len(mbt_object.y)//5:-len(mbt_object.y)//5,:len(mbt_object.x)//4],
    vmin=-0.2,
    vmax=0.2,
    cmap="seismic",
    shading="gouraud"
)

cbar = plt.colorbar(orientation="vertical")
cbar.set_label("Absolute concentration difference", fontsize=textsize)
cbar.ax.tick_params(labelsize=textsize)

plt.xlabel("Plume travel distance $x$ [m]", fontsize=textsize)
plt.ylabel('Plume cross section $y$ [m]',fontsize = textsize)
plt.ylim((-10,10))

# Change colorbar label size
plt.tick_params(axis='both', labelsize=textsize)
plt.tight_layout()
#plt.savefig('model_comparison_mbt_bio_2D.pdf')

In [ ]:
plt.figure(figsize = [3.75,2.5])

plt.pcolormesh(
    # mbt_object.x,
    # mbt_object.y,
    # diff_mbt_bio,
    mbt_object.x[:len(mbt_object.x)//4],
    mbt_object.y,
    diff_mbt_ana[:,:len(mbt_object.x)//4],
    # mbt_object.x[:len(mbt_object.x)//4],
    # mbt_object.y[len(mbt_object.y)//5:-len(mbt_object.y)//5],
    # diff_mbt_bio[len(mbt_object.y)//5:-len(mbt_object.y)//5,:len(mbt_object.x)//4],
    vmin=-0.2,
    vmax=0.2,
    cmap="seismic",
    shading="gouraud"
)

cbar = plt.colorbar(orientation="vertical")
cbar.set_label("Absolute concentration difference", fontsize=textsize)
cbar.ax.tick_params(labelsize=textsize)

plt.xlabel("Plume travel distance $x$ [m]", fontsize=textsize)
plt.ylabel('Plume cross section $y$ [m]',fontsize = textsize)
plt.ylim((-10,10))

# Change colorbar label size
plt.tick_params(axis='both', labelsize=textsize)
plt.tight_layout()
#plt.savefig('model_comparison_mbt_ana_2D.pdf')

In [ ]:
# ### Specification for Paper Plot
# xpos = 100
# time = 5*365
# models=[list_mbt[0][-1][0],list_mbt[0][0][0], list_mbt[-1][0][0],
#         list_ana[0][-1][0],list_ana[0][0][0], list_ana[-1][0][0], 
#         list_bio[0][-1][0],list_bio[0][0][0], list_bio[-1][0][0]]
# #az = 0.05, ay = 0.002, ax = 10 : [0][-1][0]
# #az = 0.05, ay = 0.2, ax = 10: [0][0][0]
# #az = 0, az = 0.2, ax = 10: [-1][0][0]

# labels=[r"mbt, $\alpha_y=0.002m$", r"mbt, $\alpha_y=0.2m$", r"mbt, $\alpha_z=0m$",
#         r"ana", r"ana", r"ana", r"bio", r"bio", r"bio"]
# linestyles=["-", "--",':', "-", "--",':', "-", "--",':']
# linewidth=2

# cmap = plt.get_cmap('tab20b')
# colors_1 = cmap.colors   # list of RGB tuples

# colors=[colors_1[0],colors_1[1],colors_1[2], 
#         colors_1[0+8],colors_1[1+8],colors_1[2+8],
#         colors_1[0+12],colors_1[1+12],colors_1[2+12]]
# textsize = 8

In [ ]:
### Plot for Paper
plt.figure(figsize = [3.75,3.2])

for i, mod in enumerate(models):
    mod.transverse(x_position=xpos, time=time, color=colors[i], lw=linewidth, linestyle=linestyles[i], label=labels[i])

plt.xlim(-20,20)
plt.ylim([0,5])
plt.legend(ncols = 3, fontsize = textsize)#, loc= 'lower left')#bbox_to_anchor=(0.3,0.55))
plt.xlabel('Plume cross section position $y$ [m]',fontsize = textsize)
plt.ylabel('Concentration $C(x_i,y,t_i)$',fontsize = textsize)
plt.title("")#Plume mass over time")
#plt.xlim([0,500])
plt.grid(True)

plt.tick_params(axis='both', labelsize=textsize)
plt.tight_layout()
#plt.savefig('sensitivity_alpha_T.pdf')


In [ ]:
df.comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 1e-4,
    relative_concentration = True,
    difference_method = "sum_mean",
    relative_diff_diff = True,
    as_percentage = False,
    colorbar_label = "normalized difference between models"
)
plt.tight_layout()
#plt.savefig('SI_Matrix_mbt_ana.pdf')

In [ ]:
df.comparison_plot(
    list_mbt, list_bio, x_disp, y_disp, z_disp,
    cutoff = 1e-4,
    relative_concentration = True,
    difference_method = "sum_mean",
    relative_diff_diff = True,
    as_percentage = False,
    colorbar_label = "normalized difference between models"
)
plt.tight_layout()
#plt.savefig('SI_Matrix_mbt_bio.pdf')